In [ ]:
import cv2
import dlib
import numpy as np
from scipy.spatial import distance as dist
from skimage.feature import hog
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
import os
import glob
from sklearn.utils import shuffle

class DrowsinessDetector:

    def __init__(self, predictor_path):
        # Initialize DrowsinessDetector with the path to the facial landmarks predictor
        self.detector = dlib.get_frontal_face_detector()
        self.predictor = dlib.shape_predictor(predictor_path)
        self.clf = svm.SVC()

    def calculate_features(self, frame):
        # Calculate facial features (HOG features, EAR, MAR) from the given frame
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = self.detector(gray)
        features = []

        for face in faces:
            shape = self.predictor(gray, face)
            shape = np.array([(shape.part(i).x, shape.part(i).y) for i in range(68)])

            left_eye = shape[36:42]
            right_eye = shape[42:48]
            mouth = shape[48:60]

            ear = (dist.euclidean(left_eye[1], left_eye[5]) + dist.euclidean(right_eye[1], right_eye[5])) / (
                    2.0 * dist.euclidean(left_eye[0], left_eye[3]))
            mar = dist.euclidean(mouth[9], mouth[1]) / dist.euclidean(mouth[0], mouth[6])

            # Extract regions of interest (ROIs) for eyes and mouth
            left_eye_roi = gray[np.min(left_eye[:, 1]):np.max(left_eye[:, 1]),
                            np.min(left_eye[:, 0]):np.max(left_eye[:, 0])]
            right_eye_roi = gray[np.min(right_eye[:, 1]):np.max(right_eye[:, 1]),
                             np.min(right_eye[:, 0]):np.max(right_eye[:, 0])]
            mouth_roi = gray[np.min(mouth[:, 1]):np.max(mouth[:, 1]),
                        np.min(mouth[:, 0]):np.max(mouth[:, 0])]

            # Check if ROIs are large enough
            if (left_eye_roi.shape[0] >= 8 and left_eye_roi.shape[1] >= 8 and
                    mouth_roi.shape[0] >= 8 and mouth_roi.shape[1] >= 8 and
                    right_eye_roi.shape[0] >= 8 and right_eye_roi.shape[1] >= 8):
                # Calculate HOG features
                hog_left_eye = hog(left_eye_roi, orientations=8, pixels_per_cell=(8, 8),
                                   cells_per_block=(2, 2), block_norm='L2-Hys')
                hog_right_eye = hog(right_eye_roi, orientations=8, pixels_per_cell=(8, 8),
                                    cells_per_block=(2, 2), block_norm='L2-Hys')
                hog_mouth = hog(mouth_roi, orientations=8, pixels_per_cell=(8, 8),
                                cells_per_block=(2, 2), block_norm='L2-Hys')
                # Append EAR and MAR to the features
                features.append(np.concatenate((hog_left_eye, hog_right_eye, hog_mouth, [ear], [mar])))

        return features

    def train(self, features, labels):
        # Flatten and pad features for SVM training
        max_length = max(len(f) for f in features)
        flattened_features = [np.concatenate((feat, np.zeros(max_length - len(feat)))) for feat in features]
        flattened_features = np.array(flattened_features)

        # Split dataset into training and test sets
        features_train, features_test, labels_train, labels_test = train_test_split(
            flattened_features, labels, test_size=0.2, random_state=42)

        # Define batch size for training SVM
        batch_size = 100
        for i in range(0, len(features_train), batch_size):
            batch_features = features_train[i:i + batch_size]
            batch_labels = labels_train[i:i + batch_size]
            self.clf.fit(batch_features, batch_labels)

        # Predict labels for the test set and print classification report
        labels_pred = self.clf.predict(features_test)
        print(classification_report(labels_test, labels_pred))

        # Define parameter grid for GridSearchCV
        param_grid = {'C': [0.1, 1, 10, 100, 1000],
                      'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
                      'kernel': ['rbf']}
        
        # Perform grid search to find the best hyperparameters
        grid = GridSearchCV(svm.SVC(), param_grid, refit=True, verbose=3)
        grid.fit(features_train, labels_train)

        # Print the best parameters and classification report
        print(grid.best_params_)
        grid_predictions = grid.predict(features_test)
        print(classification_report(labels_test, grid_predictions))

    def predict(self, features):
        # Flatten and pad features for prediction
        max_length = max(len(f) for f in features)
        flattened_features = [np.concatenate((feat, np.zeros(max_length - len(feat)))) for feat in features]
        flattened_features = np.array(flattened_features)

        # Return predictions using trained SVM classifier
        return self.clf.predict(flattened_features)


In [ ]:
# Initialize the drowsiness detector
predictor_path = r'/content/shape_predictor_68_face_landmarks.dat'
detector = DrowsinessDetector(predictor_path)

# Initialize lists to store HOG features and labels
features = []
labels = []

# Get a list of all video frame file paths
frame_files = glob.glob('/content/drive/MyDrive/train/**/*.jpg', recursive=True)

# Loop through each video frame file path
for frame_file in frame_files:
    # Read the frame
    frame = cv2.imread(frame_file)

    # Calculate features using the DrowsinessDetector
    frame_features = detector.calculate_features(frame)

    # Check if any features are obtained
    if any(feature.any() for feature in frame_features):
        # Append label to the labels list based on the frame file name
        if 'alert' in frame_file:
            labels.extend([0] * len(frame_features))  # Alert
        elif 'tired' in frame_file:
            labels.extend([1] * len(frame_features))  # Tired

        # Extend the features list
        features.extend(frame_features)

# Convert lists to numpy arrays
features = np.array(features, dtype=object)
labels = np.array(labels)

# Train the drowsiness detector
detector.train(features, labels)


In [ ]:
import pickle

# Save to file in the current working directory
pkl_filename = "pickle_model.pkl"
with open(pkl_filename, 'wb') as file:
    pickle.dump(detector, file)